# E5b v3 — seed sensitivity

Fixed E1 split (repeat 0, fold 0); runs RF importance at all k and GA/BPSO at k=50 for seeds 0–4, then evaluates LR-L2 on the untouched test fold.

In [ ]:
from pathlib import Path
import os,sys,subprocess,shutil,json,warnings
INPUT=Path('/kaggle/input/experiment-data-v3'); REPO=Path('/kaggle/working/RNA-Seq-FS-Survey'); WORK=Path('/kaggle/working/kltn_data')
COHORT='tcga_luad_lusc_primary_tumor'; SEEDS=range(5); K_RF=[1,2,3,4,5,10,25,50]; K_WRAPPER=[50]; METHODS={'rf_importance':K_RF,'ga':K_WRAPPER,'bpso':K_WRAPPER}
if REPO.exists(): shutil.rmtree(REPO)
subprocess.run(['git','clone','--branch','survey-v2-code','--single-branch','https://github.com/datachain-uit/RNA-Seq-FS-Survey.git',str(REPO)],check=True)
WORK.mkdir(parents=True,exist_ok=True)
for n in ['experiment-data-perm','experiment-data-geo-target','split_manifests']:
 src,tgt=INPUT/n,WORK/n;assert src.is_dir(),src
 if not tgt.exists():tgt.symlink_to(src,target_is_directory=True)
os.chdir(REPO);sys.path.insert(0,str(REPO))

In [ ]:
import numpy as np,pandas as pd
from threadpoolctl import threadpool_limits
from survey_v2 import config as C
from survey_v2.data import load_dataset,variance_prefilter,repeated_patient_splits
from survey_v2.methods import select
from survey_v2.classify import evaluate_panel
from survey_v2.stability import nogueira,selections_to_matrix
ds=load_dataset(WORK,COHORT); repeat,outer_seed,fold,tr,te=repeated_patient_splits(ds.y,ds.groups)[0]; assert (repeat,fold)==(0,0); pre=variance_prefilter(ds.X[tr],5000)
OUT=WORK/'results'/'survey_v3'/'E5b_seed_sensitivity'/COHORT
def save(df,p):p.parent.mkdir(parents=True,exist_ok=True);t=p.with_suffix('.tmp');df.to_csv(t,index=False);t.replace(p)
for mid,(method,ks) in enumerate(METHODS.items()):
 for seed in SEEDS:
  p=OUT/'checkpoints'/f'{method}_seed{seed}.csv'
  if p.is_file():continue
  with threadpool_limits(1): panels=select(method,ds.X[tr][:,pre],ds.y[tr],ks,seed=C.seed_for(C.dataset_seed_id(COHORT),5,2,mid,seed),n_jobs=1).panels
  rows=[]
  for k,panel in panels.items():
   cols=pre[panel]; ev=evaluate_panel(ds.X[tr][:,cols],ds.y[tr],ds.X[te][:,cols],ds.y[te],seed=seed,n_jobs=1)
   rows += [{'cohort':COHORT,'method':method,'seed':seed,'k':k,'rank':r,'gene':ds.genes[c],'auroc':ev.auroc,'bacc':ev.bacc,'mcc':ev.mcc} for r,c in enumerate(cols,1)]
  save(pd.DataFrame(rows),p);print(method,seed,flush=True)
files=sorted((OUT/'checkpoints').glob('*.csv'));assert len(files)==15,len(files)
runs=pd.concat([pd.read_csv(p) for p in files],ignore_index=True);save(runs,OUT/'e5b_seed_runs.csv')
rows=[]
for (method,k),g in runs.groupby(['method','k']):
 panels=[x.gene.tolist() for _,x in g.groupby('seed')]; rows.append({'cohort':COHORT,'method':method,'k':k,**nogueira(selections_to_matrix(panels,ds.genes))})
save(pd.DataFrame(rows),OUT/'e5b_seed_stability.csv');(OUT/'config.json').write_text(json.dumps({'experiment':'E5b_seed_sensitivity_v3','cohort':COHORT,'repeat':0,'fold':0,'seeds':list(SEEDS),'methods':METHODS},indent=2));shutil.make_archive('/kaggle/working/e5b_seed_sensitivity_v3','zip',WORK,'results/survey_v3/E5b_seed_sensitivity');print('DONE')